In [1]:
import modules.data as d
import modules.utils as u
from pathlib import Path

#### init ####
dataset_dir = Path('/home/mv18gs/Documents/GitHub/pathway_model/datasets/')
device = u.Devices().auto_set_device(drop=['cuda:4','cuda:7'])

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 77674)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 70555)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 53207)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 51647)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 23443)

# #### Device() ####
# device = cuda:3



In [ ]:
#### data ####
brca = d.TCGA(
    tcga_project = 'BRCA',
    tcga_dir = dataset_dir/'tcga',
    # type_col = 'sample_type',
    subtype_col = 'paper_BRCA_Subtype_PAM50',
    drop = ['Normal', 'Primary Tumor', 'Metastatic'],
    gene_name_path = dataset_dir/'other'/'name2ensg.csv',
    keep_noname = False,
)

kegg = d.KEGG(
    relation_filepath=dataset_dir/'other'/'relation_ohe.csv', 
    counts_data=brca,
)

dataset = d.GraphDataset(brca, kegg, kegg)
# _batch = d.get_toy_databatch(dataset, device)

('cuda:4', 'NVIDIA A100-SXM4-80GB', 81043)
('cuda:2', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:3', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:7', 'NVIDIA A100-SXM4-80GB', 81041)
('cuda:0', 'NVIDIA A100-SXM4-80GB', 78102)
('cuda:6', 'NVIDIA A100-SXM4-80GB', 53143)
('cuda:1', 'NVIDIA A100-SXM4-80GB', 46635)
('cuda:5', 'NVIDIA A100-SXM4-80GB', 40625)

# #### Device() ####
# device = cuda:2

# #### KEGG() ####
# _orig_kwargs             5                        dict
# relation                 (75939, 19)              DataFrame
# ensg                     4373                     list
# pathway_labels           305                      list
# edge_index               (2, 32464)               Tensor (cuda:2)
# edge_attr                (32464, 16)              Tensor (cuda:2)
# edge_labels              16                       list
# pathway_index            (4373, 305)              Tensor (cuda:2)

# #### TCGA() ####
# _orig_kwargs             9                        dict
# counts_path            

In [2]:
from modules.layers import MultiheadSetPooling
from modules.loss import MultiLoss, KLDLoss
from modules.model import BaseAutoencoder
from modules.norm import LogCounts
from modules.train import Experiment, grid
from modules.trainers import ReconstrTrainer

from functools import partial
from torch_geometric.nn import GCNConv, GATConv
import torch.nn as nn

In [ ]:
## Model

model_template = partial(
    BaseAutoencoder,
    dataset = dataset,
    embed_dim = 128,
    num_heads=4,
    hidden_dims = 2,
    method='set',
    mlp=False,
    variational=True,

    # layers
    norm_class=LogCounts,
    encoder_class=nn.Linear,
    pooling_class=MultiheadSetPooling,

    # layer params
    act_fn=nn.ReLU, 
    norm_fn='layer', 
    end_fn=False,

    # kwargs
    norm_kwargs = {'libnorm':True, 'znorm':True, 'learnable':True},
)

model_grid = grid(
    model_template,
)

## Trainer

trainer_template = partial(
    ReconstrTrainer,
    lr=1e-3,
    norm_class=LogCounts,
    out_keys={'input':'x_t_pred', 'target':'x_t', 'mu':'z_mu', 'logvar':'z_logvar'},
    loss_class=MultiLoss,
)

trainer_grid = grid(
    trainer_template,
    loss_kwargs=[
        {'loss_classes':[nn.MSELoss, KLDLoss], 'loss_weights': (1,1e-4), 'ema_norm':True},
        {'loss_classes':[nn.MSELoss, KLDLoss], 'loss_weights': (1,1e-5), 'ema_norm':True},
        {'loss_classes':[nn.MSELoss, KLDLoss], 'loss_weights': (1,1e-6), 'ema_norm':True},
        {'loss_classes':[nn.MSELoss, KLDLoss], 'loss_weights': (1,1e-7), 'ema_norm':True},
    ]
)

## Experiment

expt = Experiment(
    num_trials=10,
    num_epochs=200,
    dataset=dataset,
    device=device,
    batch_size=128
)

expt.add_grid(
    model_grid=model_grid,
    trainer_grid=trainer_grid
)

print(len(expt.configs))

4


In [7]:
expt.run_experiment(
    'reconstr_grid4b',
    report_metrics=['loss','kld','rmse','mae','r2'],
    save_csv=True,
    save_params=True,
    save_values=True,
    verbose=False,
)

  0%|          | 0/10 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]

  0%|          | 0/200 [00:00<?, ?it/s]